# S7-s1 Limpieza y transformacion

In [1]:
import pandas as pd
df = pd.read_csv('NovaMarket_datos_crudos.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 640000 entries, 0 to 639999
Data columns (total 20 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   id_pedido                  640000 non-null  object 
 1   id_cliente                 640000 non-null  object 
 2   fecha_pedido               640000 non-null  object 
 3   canal_compra               640000 non-null  object 
 4   metodo_pago                640000 non-null  object 
 5   ciudad_tienda              610072 non-null  object 
 6   codigo_postal              640000 non-null  int64  
 7   categoria_producto         640000 non-null  object 
 8   producto                   640000 non-null  object 
 9   precio_unitario            618360 non-null  float64
 10  unidades_vendidas          640000 non-null  int64  
 11  monto_compra               618360 non-null  float64
 12  edad_cliente               640000 non-null  int64  
 13  correo_cliente             53

### 1. Inconsistencia de texto

In [2]:
# Corregir cada fuente en pandas
# 1. Unificar mayúsculas y minúsculas
df["ciudad_tienda"] = df["ciudad_tienda"].str.strip().str.title()

# 2. Quitar espacios extra al inicio, al final y dobles en medio
df["ciudad_tienda"] = df["ciudad_tienda"].str.replace(r"\s+", " ", regex=True).str.strip()

# 3. Verificar si quedaron variantes por tildes
df["ciudad_tienda"].unique()

array(['Bogota', 'Mzl', 'Cúcuta', 'Cucuta', 'Bucaramanga', 'Mde',
       'Pereira', 'Medellín', 'Manizales', 'Cali', 'Santa Marta',
       'Ibagué', 'Villavicencio', 'B/Manga', 'Ctg', 'Bogotá',
       'Barranquilla', 'Baq', 'B/Quilla', nan, 'Ibague', 'Cartagena',
       'Santamarta', 'Medellin', 'Cartagena De Indias', 'Sta. Marta',
       'Bog', 'Medellin, Ant.', 'Bogota D.C.', 'Bga', 'Smr',
       'Santiago De Cali', 'Bogota Dc', 'Manizales, Caldas'], dtype=object)

In [3]:
# Para tildes se mapea a mano
correcciones_ciudad = {
    "Bogota": "Bogotá",
    "Medellin": "Medellín",
}
df["ciudad_tienda"] = df["ciudad_tienda"].replace(correcciones_ciudad)

In [4]:
# Para categorías con abreviaturas distintas, como "Electro" por "Electrónica", se usa la misma técnica de mapeo:
correcciones_categoria = {
    "Electro": "Electrónica",
    "ELECTRONICA": "Electrónica",
    "Hogar y Deco": "Hogar",
}
df["categoria_producto"] = df["categoria_producto"].str.strip().replace(correcciones_categoria)

In [5]:
# Como encontrar todas las variantes antes de corregir
df["ciudad_tienda"].unique()

array(['Bogotá', 'Mzl', 'Cúcuta', 'Cucuta', 'Bucaramanga', 'Mde',
       'Pereira', 'Medellín', 'Manizales', 'Cali', 'Santa Marta',
       'Ibagué', 'Villavicencio', 'B/Manga', 'Ctg', 'Barranquilla', 'Baq',
       'B/Quilla', nan, 'Ibague', 'Cartagena', 'Santamarta',
       'Cartagena De Indias', 'Sta. Marta', 'Bog', 'Medellin, Ant.',
       'Bogota D.C.', 'Bga', 'Smr', 'Santiago De Cali', 'Bogota Dc',
       'Manizales, Caldas'], dtype=object)

### 2. Inconsistencia formatos

In [6]:
# Formatos de texto que pd.to_datetime() no reconoce solo
# Después de este reemplazo, "5-ago-2026" queda como "5-08-2026", un formato que pd.to_datetime() sí puede interpretar junto con los demás.
meses_es_a_num = {
    "ene": "01", "feb": "02", "mar": "03", "abr": "04",
    "may": "05", "jun": "06", "jul": "07", "ago": "08",
    "sep": "09", "oct": "10", "nov": "11", "dic": "12",
}
for mes_es, mes_num in meses_es_a_num.items():
    df["fecha_pedido"] = df["fecha_pedido"].str.replace(f"-{mes_es}-", f"-{mes_num}-", regex=False)

In [7]:
# Convierte la columna a un tipo de dato de fecha real con pd.to_datetime(), en vez de dejarla como texto:
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"], dayfirst=True, errors="coerce")

In [8]:
# Verificando cuántas fechas quedaron sin convertir
df["fecha_pedido"].isnull().sum()

np.int64(223053)

In [9]:
# Categorías equivalentes: cuando dos palabras distintas significan lo mismo
correcciones_canal = {
    "Aplicación móvil": "App",
    "APP": "App",
}
df["canal_compra"] = df["canal_compra"].str.strip().replace(correcciones_canal)

### 3. Duplicados deteccion

In [10]:
# Duplicados exactos: filas idénticas letra por letra
# Cuenta cuántas filas están duplicadas por completo
df.duplicated().sum()

# Muestra esas filas para revisarlas antes de decidir qué hacer
df[df.duplicated(keep=False)]

,id_pedido,id_cliente,fecha_pedido,canal_compra,metodo_pago,ciudad_tienda,codigo_postal,categoria_producto,producto,precio_unitario,unidades_vendidas,monto_compra,edad_cliente,correo_cliente,nivel_satisfaccion,nivel_lealtad,comentario_cliente,peso_pedido_kg,fecha_actualizacion_stock,fuga_cliente
4,P247920,C163018,2025-05-01,Web,Efectivo,Bucaramanga,6800112,Hogar,Juego de sábanas,222400.0,1,222400.0,60,cliente147920@novamarket.com.co,Alto,Plata,NaN,12.3,2026-07-13,0
17,P583009,C122999,NaT,Tienda,PayPal,Bucaramanga,6800101,Hogar,Set de cuchillos,210100.0,2,420200.0,23,cliente483009@hotmail.com,Bajo,Oro,Pésima experiencia con el empaque,8.5,2026-05-26,1
20,P551733,C149462,2025-08-31,App,Tarjeta,Pereira,6600102,Electrónica,Teclado mecánico,610600.0,2,1221200.0,45,cliente451733@novamarket.com.co,Bajo,Plata,Buena atención pero demoró mucho,13.1,2023-12-07,0
31,P418846,C51065,2026-01-14,Web,PayPal,Bogotá,7600101,Juguetería,Muñeca articulada,610500.0,3,1831500.0,44,cliente318846@gmail.com,Alto,Plata,Me costó encontrar el producto en la app,13.0,2026-06-04,0
36,P332336,C46793,2026-01-15,app,Tarjeta,Bogotá,1100119,Deportes,Bicicleta estática,158700.0,3,476100.0,49,cliente232336.gmail.com,Medio,Bronce,"Excelente calidad, volvería a comprar",9.0,2026-05-03,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
639942,P423892,C119977,NaT,Marketplace,PayPal,Cartagena,1300110,HOGAR,Juego de sábanas,568500.0,1,568500.0,45,NaN,Medio,Bronce,Cumplió con lo prometido,13.9,2026-06-25,1
639948,P592410,C65292,2025-10-16,App,PayPal,Medellín,500104,juguetería,Carro a control remoto,564500.0,1,564500.0,-5,cliente492410@hotmail.com,Alto,Plata,No era lo que esperaba,10.8,2026-03-27,0
639954,P343844,C35271,NaT,tienda,PayPal,Cúcuta,5400101,Deportes,Pesas ajustables,-590700.0,3,-1772100.0,59,cliente243844@gmail.com,Bajo,Oro,"Excelente calidad, volvería a comprar",8.8,2026-03-04,0
639981,P180066,C92020,2025-01-26,Marketplace,Transferencia,NaN,5400101,Deportes,Bicicleta estática,435600.0,1,435600.0,28,cliente80066@outlook.com,Alto,Bronce,El producto llegó dañado,8.2,2026-06-10,0


In [11]:
# bDuplicados casi exactos: la misma entidad, con una pequeña diferencia
# Después de estandarizar nombre_cliente y correo_cliente (lección anterior),
# revisa duplicados usando solo las columnas que identifican a un cliente
df.duplicated(subset=["correo_cliente"], keep=False).sum()

np.int64(138688)

In [12]:
# Verificando cuántos duplicados encontraste, antes de borrar nada
print(f"Duplicados exactos: {df.duplicated().sum()}")
print(f"Duplicados por correo: {df.duplicated(subset=['correo_cliente'], keep=False).sum()}")

Duplicados exactos: 20000
Duplicados por correo: 138688


### 4. Duplicados resolucion

In [13]:
# Estrategia 1: eliminación simple, quedarte con una copia
# Elimina duplicados exactos, se queda con la primera aparición
df = df.drop_duplicates(keep="first")

# Elimina duplicados por correo, conservando el registro más reciente
df = df.sort_values("fecha_pedido").drop_duplicates(subset=["correo_cliente"], keep="last")

In [14]:
# Estrategia 2: combinar información antes de eliminar
# Agrupa por correo y, para cada columna, se queda con el primer valor no vacío
df_combinado = df.groupby("correo_cliente", as_index=False).first()

In [15]:
# Documentando cada decisión
filas_antes = df.shape[0]
df = df.drop_duplicates(keep="first")
filas_despues = df.shape[0]
print(f"Se eliminaron {filas_antes - filas_despues} filas duplicadas.")

Se eliminaron 0 filas duplicadas.


# Ejercicio clase

In [16]:
import pandas as pd
df = pd.read_csv("S04_PD_GomezGuerrero_DatasetImputacionInicial.csv")

In [17]:
for col in ["ciudad_tienda", "categoria_producto", "canal_compra", "metodo_pago"]: print(col, sorted(df[col].dropna().unique()))

ciudad_tienda ['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Cucuta', 'Ibague', 'Manizales', 'Medellin', 'Pereira', 'Santa Marta', 'Villavicencio']
categoria_producto [' Belleza', ' Deportes', ' Electrónica', ' Hogar', ' Juguetería', ' Moda', 'BELLEZA', 'Belleza', 'Belleza ', 'DEPORTES', 'Deportes', 'Deportes ', 'ELECTRÓNICA', 'Electrónica', 'Electrónica ', 'HOGAR', 'Hogar', 'Hogar ', 'JUGUETERÍA', 'Juguetería', 'Juguetería ', 'MODA', 'Moda', 'Moda ', 'belleza', 'deportes', 'electronica', 'electrónica', 'hogar', 'jugueteria', 'juguetería', 'moda']
canal_compra [' App', ' Marketplace', ' Tienda', ' Web', 'APP', 'App', 'App ', 'MARKETPLACE', 'Marketplace', 'Marketplace ', 'TIENDA', 'Tienda', 'Tienda ', 'WEB', 'Web', 'Web ', 'app', 'marketplace', 'tienda', 'web']
metodo_pago [' Efectivo', ' PayPal', ' Tarjeta', ' Transferencia', 'EFECTIVO', 'Efectivo', 'Efectivo ', 'PAYPAL', 'PayPal', 'PayPal ', 'TARJETA', 'TRANSFERENCIA', 'Tarjeta', 'Tarjeta ', 'Transferencia', 'Transfer

In [18]:
df[col].value_counts()

metodo_pago
Transferencia     124051
Tarjeta           123624
PayPal            123566
Efectivo          123459
paypal             12740
transferencia      12545
efectivo           12535
tarjeta            12427
 Efectivo           6363
Tarjeta             6353
 Transferencia      6340
Transferencia       6327
 Tarjeta            6315
EFECTIVO            6299
PAYPAL              6235
 PayPal             6202
Efectivo            6192
TARJETA             6188
PayPal              6179
TRANSFERENCIA       6060
Name: count, dtype: int64

- Pienso que las columnas en categora_producto que representan lo mismo es Electronica, ELECTRONICA; BELLEZA, belleza; DEPORTES, deportes; HOGAR, hogar; Moda, moda
- En canal_compra: App, APP; MARKETPLACE, Marketplace; TIENDA, Tienda; WEB, Web
- En metodo_pago: Efectvi, EFECTIVO; PAYPAL, paypal; TARJETA, Tarjeta; TRANSFERENCIA, Transferencia

## Unifica mayúsculas, minúsculas y espacios

In [19]:
df["ciudad_tienda"] = df["ciudad_tienda"].str.strip().str.title() #strip() para quitar espacios y title() para mayusculas
df["categoria_producto"] = df["categoria_producto"].str.strip().str.title()
df["canal_compra"] = df["canal_compra"].str.strip().str.title()
df["metodo_pago"] = df["metodo_pago"].str.strip().str.title()

In [20]:
df[col].value_counts()

metodo_pago
Transferencia    155323
Paypal           154922
Tarjeta          154907
Efectivo         154848
Name: count, dtype: int64

In [21]:
df.shape[0]

620000

## Diccionarios de corrección

In [22]:
correcciones_ciudad = {"Bogota": "Bogotá", "Medellin": "Medellín"}
correcciones_categoria = {"Electronica": "Electrónica", "Jugueteria": "Juguetería"}
correcciones_pago = {"Paypal": "PayPal"}

In [23]:
df["ciudad_tienda"] = df["ciudad_tienda"].replace(correcciones_ciudad)
df["categoria_producto"] = df["categoria_producto"].replace(correcciones_categoria)
df["metodo_pago"] = df["metodo_pago"].replace(correcciones_pago)

In [24]:
df["ciudad_tienda"].unique()

array(['Bogotá', 'Manizales', 'Cucuta', 'Bucaramanga', 'Medellín',
       'Pereira', 'Cali', 'Santa Marta', 'Ibague', 'Villavicencio',
       'Cartagena', 'Barranquilla'], dtype=object)

In [25]:
df["categoria_producto"].unique()

array(['Moda', 'Electrónica', 'Juguetería', 'Deportes', 'Hogar',
       'Belleza'], dtype=object)

In [26]:
df["canal_compra"].unique()

array(['Tienda', 'Web', 'App', 'Marketplace'], dtype=object)

In [27]:
df["metodo_pago"].unique()

array(['Efectivo', 'PayPal', 'Transferencia', 'Tarjeta'], dtype=object)

## Convierte fecha_pedido a un formato de fecha real

In [28]:
meses_es_a_num = {"ene": "01", "feb": "02", "mar": "03", "abr": "04", "may": "05", "jun": "06", "jul": "07", "ago": "08", "sep": "09", "oct": "10", "nov": "11", "dic": "12"}

In [29]:
for mes_es, mes_num in meses_es_a_num.items(): df["fecha_pedido"] = df["fecha_pedido"].str.replace(f"-{mes_es}-", f"-{mes_num}-", regex=False)

In [30]:
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"], dayfirst=True, errors="coerce")

In [31]:
df["fecha_pedido"].isnull().sum()

np.int64(216162)

In [32]:
df[df["fecha_pedido"].isnull()]

,id_pedido,id_cliente,fecha_pedido,canal_compra,metodo_pago,ciudad_tienda,codigo_postal,categoria_producto,producto,precio_unitario,...,peso_pedido_kg,fecha_actualizacion_stock,fuga_cliente,ciudad_tienda_imputada,satisfaccion_sin_dato,correo_sin_dato,sin_comentario,precio_sin_dato,monto_sin_dato,fecha_pedido_sin_dato
1,P677836,C84260,NaT,Web,Efectivo,Manizales,1700101,Electrónica,Parlante bluetooth,195700.0,...,12.9,2026-06-10,0,False,False,False,False,False,False,False
2,P387869,C164166,NaT,Tienda,PayPal,Cucuta,5400101,Juguetería,Rompecabezas 1000 piezas,597600.0,...,14.3,2026-02-26,0,False,False,False,False,False,False,False
5,P465514,C102032,NaT,Tienda,Efectivo,Medellín,500102,Deportes,Balón de fútbol,626800.0,...,12.7,2027-07-21,0,False,False,False,False,False,False,False
9,P622879,C155042,NaT,Marketplace,Efectivo,Cali,7600101,Electrónica,Audífonos inalámbricos,-124300.0,...,15.0,2026-05-08,0,False,False,False,False,False,False,False
15,P332138,C76833,NaT,Tienda,Tarjeta,Bucaramanga,6800108,Electrónica,Monitor 24 pulgadas,396200.0,...,8.9,2026-07-09,1,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619989,P683071,C28408,NaT,Marketplace,Transferencia,Bucaramanga,6800112,Deportes,Colchoneta de yoga,NaN,...,1237.6,2026-07-28,0,False,False,False,False,True,True,False
619990,P257363,C23142,NaT,Web,Tarjeta,Medellín,500101,Electrónica,Monitor 24 pulgadas,416400.0,...,12.5,2026-05-26,0,False,False,False,True,False,False,False
619991,P711828,C133738,NaT,App,PayPal,Cali,7600129,Juguetería,Carro a control remoto,220200.0,...,9.5,2026-07-15,1,False,False,False,False,False,False,False
619993,P543968,C141410,NaT,App,Transferencia,Villavicencio,5000101,Electrónica,Monitor 24 pulgadas,111400.0,...,12.0,2026-06-18,0,False,False,True,False,False,False,False


## Detecta los duplicados exactos

In [33]:
df.duplicated().sum()

np.int64(0)

## Detecta los duplicados por correo_cliente

In [34]:
df.duplicated(subset=["correo_cliente"], keep=False).sum()

np.int64(102000)

In [35]:
df[df.duplicated(subset=["correo_cliente"], keep=False)].sort_values("correo_cliente")

,id_pedido,id_cliente,fecha_pedido,canal_compra,metodo_pago,ciudad_tienda,codigo_postal,categoria_producto,producto,precio_unitario,...,peso_pedido_kg,fecha_actualizacion_stock,fuga_cliente,ciudad_tienda_imputada,satisfaccion_sin_dato,correo_sin_dato,sin_comentario,precio_sin_dato,monto_sin_dato,fecha_pedido_sin_dato
12,P500774,C159046,2026-01-01,App,Efectivo,Bucaramanga,6800101,Moda,Camisa de lino,344500.0,...,13.4,2026-07-05,1,False,False,True,False,False,False,False
411963,P212591,C163692,2025-07-08,Marketplace,PayPal,Ibague,7300107,Deportes,Pesas ajustables,31300.0,...,13.9,2026-05-31,0,False,False,True,False,False,False,False
411959,P472433,C102599,NaT,App,PayPal,Ibague,7300101,Deportes,Bicicleta estática,649000.0,...,14.8,2026-04-20,0,False,False,True,False,False,False,False
411958,P565329,C31226,2026-04-15,Web,PayPal,Cali,7600101,Deportes,Colchoneta de yoga,170400.0,...,10.8,2026-02-20,0,False,False,True,False,False,False,False
411934,P583695,C93835,2025-12-25,Web,Tarjeta,Manizales,1700103,Moda,Bufanda de lana,157400.0,...,11.7,2026-03-02,0,True,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205274,P567520,C34878,2025-10-18,App,Efectivo,Medellín,500118,Hogar,Juego de sábanas,241200.0,...,11.9,2026-07-14,1,False,True,True,False,False,False,False
205270,P284896,C89762,NaT,App,Transferencia,Manizales,1700103,Juguetería,Rompecabezas 1000 piezas,445900.0,...,14.9,2026-05-18,0,False,False,True,False,False,False,False
205269,P520253,C40518,2025-06-28,App,Tarjeta,Santa Marta,4700101,Juguetería,Set de bloques,97500.0,...,13.7,2026-05-27,1,False,True,True,False,False,False,False
205261,P109906,C27535,2025-06-04,App,Tarjeta,Ibague,7300101,Hogar,Lámpara de escritorio,148400.0,...,8.3,2026-02-26,0,False,False,True,False,False,False,False


In [36]:
# Muestra solo el correo y el id para ver los grupos juntos
df[df.duplicated(subset=["correo_cliente"], keep=False)].sort_values("correo_cliente")[["correo_cliente", "id_cliente", "id_pedido"]].head(10)

,correo_cliente,id_cliente,id_pedido
12,desconocido,C159046,P500774
411963,desconocido,C163692,P212591
411959,desconocido,C102599,P472433
411958,desconocido,C31226,P565329
411934,desconocido,C93835,P583695
411927,desconocido,C117587,P250451
411926,desconocido,C131100,P606351
411923,desconocido,C120564,P611918
411920,desconocido,C125868,P712379
411919,desconocido,C163574,P659975


In [37]:
# Filtra correos repetidos excluyendo la palabra "desconocido"
df_sin_desconocido = df[df["correo_cliente"] != "desconocido"]

# Muestra los grupos de correos repetidos reales
df_sin_desconocido[
    df_sin_desconocido.duplicated(subset=["correo_cliente"], keep=False)
].sort_values("correo_cliente")[["correo_cliente", "id_cliente", "id_pedido"]].head(10)

,correo_cliente,id_cliente,id_pedido


## Duplicados por correo encontrados

Aparecen 102.000 filas repetidas, pero esto no se debe a correos repetidos ni duplicados, sino a que no tienen correo electronico y el dataset les puso como "desconocido" entonces por eso aparecen esos registros "duplicados"

## Regla de conservación de duplicados

Conservare todas las filas de duplicados por correo, ya que no son correos que esten duplicados sino qu eno tienen correo, por lo que son sumamente necesarios para los analisis con los datos

In [38]:
filas_antes = df.shape[0]

In [39]:
df.to_csv("S06_PD_GomezGuerrero_DatasetDepurado.csv", index=False)

In [40]:
pd.read_csv("S06_PD_GomezGuerrero_DatasetDepurado.csv")

,id_pedido,id_cliente,fecha_pedido,canal_compra,metodo_pago,ciudad_tienda,codigo_postal,categoria_producto,producto,precio_unitario,...,peso_pedido_kg,fecha_actualizacion_stock,fuga_cliente,ciudad_tienda_imputada,satisfaccion_sin_dato,correo_sin_dato,sin_comentario,precio_sin_dato,monto_sin_dato,fecha_pedido_sin_dato
0,P505195,C45650,2025-11-15,Tienda,Efectivo,Bogotá,1100103,Moda,Chaqueta impermeable,28700.0,...,14.9,2026-04-17,0,False,False,False,False,False,False,False
1,P677836,C84260,NaN,Web,Efectivo,Manizales,1700101,Electrónica,Parlante bluetooth,195700.0,...,12.9,2026-06-10,0,False,False,False,False,False,False,False
2,P387869,C164166,NaN,Tienda,PayPal,Cucuta,5400101,Juguetería,Rompecabezas 1000 piezas,597600.0,...,14.3,2026-02-26,0,False,False,False,False,False,False,False
3,P481069,C98406,2025-06-27,App,Transferencia,Cucuta,5400103,Deportes,Colchoneta de yoga,322900.0,...,9.5,2026-05-14,0,False,False,False,False,False,False,False
4,P247920,C163018,2025-05-01,Web,Efectivo,Bucaramanga,6800112,Hogar,Juego de sábanas,222400.0,...,12.3,2026-07-13,0,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619995,P264888,C145427,NaN,Marketplace,Transferencia,Bogotá,1100102,Electrónica,Audífonos inalámbricos,598700.0,...,14.6,2026-08-01,0,False,False,False,False,False,False,False
619996,P709314,C66535,2025-02-07,App,Transferencia,Santa Marta,4700102,Deportes,Bicicleta estática,504700.0,...,8.0,2023-01-13,1,False,False,False,False,False,False,False
619997,P592869,C42878,2025-11-29,Web,PayPal,Villavicencio,5000101,Deportes,Bicicleta estática,338000.0,...,14.3,2026-03-03,0,False,False,False,False,False,False,False
619998,P131880,C99715,2026-03-13,App,PayPal,Cucuta,5400101,Electrónica,Teclado mecánico,619200.0,...,12.5,2026-07-21,0,False,True,False,True,False,False,False


In [41]:
df[col].value_counts()

metodo_pago
Transferencia    155323
PayPal           154922
Tarjeta          154907
Efectivo         154848
Name: count, dtype: int64

In [42]:
df_depurado = pd.read_csv("S06_PD_GomezGuerrero_DatasetDepurado.csv")

In [43]:
df_depurado.duplicated().sum()

np.int64(0)

In [44]:
print(df_depurado["canal_compra"].unique())
print(df_depurado["metodo_pago"].unique())
print(df_depurado["categoria_producto"].unique())

['Tienda' 'Web' 'App' 'Marketplace']
['Efectivo' 'PayPal' 'Transferencia' 'Tarjeta']
['Moda' 'Electrónica' 'Juguetería' 'Deportes' 'Hogar' 'Belleza']


In [45]:
pd.to_datetime(df_depurado["fecha_pedido"])

0        2025-11-15
1               NaT
2               NaT
3        2025-06-27
4        2025-05-01
            ...    
619995          NaT
619996   2025-02-07
619997   2025-11-29
619998   2026-03-13
619999   2025-08-01
Name: fecha_pedido, Length: 620000, dtype: datetime64[ns]

In [46]:
df.columns

Index(['id_pedido', 'id_cliente', 'fecha_pedido', 'canal_compra',
       'metodo_pago', 'ciudad_tienda', 'codigo_postal', 'categoria_producto',
       'producto', 'precio_unitario', 'unidades_vendidas', 'monto_compra',
       'edad_cliente', 'correo_cliente', 'nivel_satisfaccion', 'nivel_lealtad',
       'comentario_cliente', 'peso_pedido_kg', 'fecha_actualizacion_stock',
       'fuga_cliente', 'ciudad_tienda_imputada', 'satisfaccion_sin_dato',
       'correo_sin_dato', 'sin_comentario', 'precio_sin_dato',
       'monto_sin_dato', 'fecha_pedido_sin_dato'],
      dtype='object')